# Urban Heat & Cooling-Priority Mapping — Track A / S6: Rank-Impact Test

**NUS-ISS Practice Module, Week 2.** Fills the C3 ablation row: rebuilds the
cooling-priority score once per heat-layer variant (Exposure = that variant's
LST, everything else frozen) and reports the decision-impact metrics the
eval rules require. One notebook, run top to bottom:

1. **Setup** — install deps, mount Drive (no Earth Engine needed here — this
   notebook is pure pandas/sklearn on the CSV `gee_heat_variants.ipynb` already exported)
2. **RI.1** — config (paths, weighting scheme, toy-mode switch)
3. **RI.2** — load + join heat variants with sensitivity/adaptive pillars
4. **RI.3** — score construction (PCA refit per variant / equal-weight)
5. **RI.4** — ablation metrics (RMSE vs held-out, Spearman, top-20 overlap)
6. **RI.5** — run: build the rank-impact table
7. **RI.6** — verdict
8. **RI.7 (optional)** — rerun under the other weighting scheme (sensitivity check)
9. **RI.8** — save results to Drive

Run cells in order. RI.5 depends on RI.1-RI.4.

---


# SETUP — run once per session

## Setup 1 — Install dependencies

In [1]:
# --- SETUP CELL 1: Install all dependencies used in this notebook ----------
!pip install -q pandas numpy scipy scikit-learn


## Setup 2 — Mount Google Drive

In [2]:
# --- SETUP CELL 2: Mount Google Drive (reads the gee_heat_variants.ipynb export) --
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted at /content/drive")


Mounted at /content/drive
Drive mounted at /content/drive


## Setup 3 — Initialize shared results tracker

In [3]:
# --- SETUP CELL 3: Initialize shared results tracker ------------------------
# Same pattern as gee_heat_variants.ipynb's track_a_results / Week-1's gate_results.
rank_impact_results = {}
print("rank_impact_results initialized — populated by the RI.6 verdict cell.")


rank_impact_results initialized — populated by the RI.6 verdict cell.


---
# RI — Rank-Impact Test (C3 ablation row)


## RI.1 — Config

`HEAT_CSV_PATH` below matches the `EXPORT_FOLDER` / `EXPORT_FILE_PREFIX` used
in `gee_heat_variants.ipynb` — if you changed those there, change them here too.


In [4]:
# --- RI CELL 1: Config -------------------------------------------------------
# Must match EXPORT_FOLDER / EXPORT_FILE_PREFIX from gee_heat_variants.ipynb.
EXPORT_FOLDER = "urban_heat_sg"
EXPORT_FILE_PREFIX = "heat_variants_subzone"
HEAT_CSV_PATH = f"/content/drive/MyDrive/{EXPORT_FOLDER}/{EXPORT_FILE_PREFIX}.csv"

# Pillar tables — point these at your real files once they exist.
SENSITIVITY_CSV_PATH = f"/content/drive/MyDrive/{EXPORT_FOLDER}/sensitivity_pillar.csv"
ADAPTIVE_CSV_PATH = f"/content/drive/MyDrive/{EXPORT_FOLDER}/adaptive_capacity_pillar.csv"

# Independent LST ground truth (e.g. NEA stations joined to subzones, per Week-1 G2).
# Set to None to skip RMSE (the rest of the table still runs).
#HELDOUT_CSV_PATH = None
HELDOUT_CSV_PATH = f"/content/drive/MyDrive/{EXPORT_FOLDER}/nea_heldout_lst.csv"

# TOY_MODE generates seeded placeholder sensitivity/adaptive pillars so you can
# test the wiring today. Flip to False once SENSITIVITY_CSV_PATH / ADAPTIVE_CSV_PATH
# point at real files — the notebook will refuse to run FALSE mode against
# missing/placeholder files rather than silently using toy data.
TOY_MODE = False

WEIGHTING = "pca"   # "pca" (primary) or "equal" (sensitivity check, per eval rules)
TOP_N = 20
SEED = 42            # pinned, per eval rules
REFERENCE_VARIANT = "lst_native30"
VARIANT_COLUMNS = ["lst_native30", "lst_bicubic10", "lst_regress10"]

import numpy as np
np.random.seed(SEED)

print(f"HEAT_CSV_PATH: {HEAT_CSV_PATH}")
print(f"TOY_MODE: {TOY_MODE} | WEIGHTING: {WEIGHTING} | TOP_N: {TOP_N} | SEED: {SEED}")

import os
if not os.path.exists(HEAT_CSV_PATH):
    raise FileNotFoundError(
        f"{HEAT_CSV_PATH} not found. Run gee_heat_variants.ipynb's export task first, "
        f"wait for it to show COMPLETED at https://code.earthengine.google.com/tasks, "
        f"then re-run this cell."
    )
print("✅ Heat-variants CSV found.")

if not TOY_MODE:
    for label, path in [("SENSITIVITY_CSV_PATH", SENSITIVITY_CSV_PATH), ("ADAPTIVE_CSV_PATH", ADAPTIVE_CSV_PATH)]:
        if not os.path.exists(path):
            raise FileNotFoundError(
                f"TOY_MODE is False but {label} ({path}) doesn't exist. "
                f"Either supply the real file or set TOY_MODE = True to test wiring."
            )
    print("✅ Sensitivity/adaptive pillar files found.")


HEAT_CSV_PATH: /content/drive/MyDrive/urban_heat_sg/heat_variants_subzone.csv
TOY_MODE: False | WEIGHTING: pca | TOP_N: 20 | SEED: 42
✅ Heat-variants CSV found.
✅ Sensitivity/adaptive pillar files found.


## RI.2 — Load + join

Same "count what got dropped, don't assume" discipline as the Week-1/S2
notebooks' join steps — a subzone_id mismatch here silently shrinks your
scored set instead of erroring.


In [5]:
# --- RI CELL 2: normalize() + toy pillar generator ---------------------------
import pandas as pd

def normalize(s: pd.Series) -> pd.Series:
    """Min-max normalize. Identical to Week-1 G3.10's normalize()."""
    if s.max() == s.min():
        return s * 0
    return (s - s.min()) / (s.max() - s.min())


def make_toy_pillars(subzone_ids):
    """Seeded placeholder pillars, same spirit as G3.10's dummy sensitivity.
    NOT real data — for wiring tests only."""
    rng = np.random.default_rng(SEED)
    sensitivity = pd.DataFrame({
        "subzone_id": subzone_ids,
        "sensitivity_raw": rng.uniform(0.3, 1.0, size=len(subzone_ids)),
    })
    adaptive = pd.DataFrame({
        "subzone_id": subzone_ids,
        "greenery_fraction": rng.uniform(0.05, 0.6, size=len(subzone_ids)),
    })
    return sensitivity, adaptive

print("Helpers defined.")


Helpers defined.


In [6]:
# --- RI CELL 3: Load + join ---------------------------------------------------
heat = pd.read_csv(HEAT_CSV_PATH)
missing_variant_cols = set(VARIANT_COLUMNS) - set(heat.columns)
if missing_variant_cols:
    raise ValueError(f"HEAT_CSV_PATH is missing columns {missing_variant_cols}. "
                      f"Expected subzone_id + {VARIANT_COLUMNS}.")
print(f"Loaded {len(heat)} subzones from heat-variants CSV.")

if TOY_MODE:
    print("⚠️  TOY_MODE: generating seeded placeholder sensitivity/adaptive pillars. "
          "Not real data — do not report numbers from this run.")
    sensitivity, adaptive = make_toy_pillars(heat["subzone_id"])
else:
    sensitivity = pd.read_csv(SENSITIVITY_CSV_PATH)
    adaptive = pd.read_csv(ADAPTIVE_CSV_PATH)

if "sensitivity_raw" not in sensitivity.columns:
    raise ValueError("Sensitivity table must have a 'sensitivity_raw' column.")
if "greenery_fraction" not in adaptive.columns:
    raise ValueError("Adaptive table must have a 'greenery_fraction' column.")

df = heat.merge(sensitivity, on="subzone_id", how="inner")
df = df.merge(adaptive, on="subzone_id", how="inner")

n_dropped = len(heat) - len(df)
print(f"\nJoin summary: {len(heat)} subzones in heat CSV -> {len(df)} retained after pillar join")
if n_dropped:
    print(f"⚠️  {n_dropped} subzones dropped — check subzone_id spelling/casing across files "
          f"before trusting the rank-impact numbers below.")
else:
    print("✅ No subzones dropped in the pillar join.")

heldout = None
if HELDOUT_CSV_PATH is not None:
    heldout = pd.read_csv(HELDOUT_CSV_PATH)
    if "lst_heldout_c" not in heldout.columns:
        raise ValueError("Held-out table must have an 'lst_heldout_c' column.")
    missing_ho = set(df["subzone_id"]) - set(heldout["subzone_id"])
    if missing_ho:
        print(f"⚠️  {len(missing_ho)} subzones missing from held-out LST table — "
              f"RMSE computed on the {len(df) - len(missing_ho)}-subzone overlap only.")
else:
    print("ℹ️  No HELDOUT_CSV_PATH set — lst_rmse_heldout will print as NaN for all variants.")


Loaded 332 subzones from heat-variants CSV.

Join summary: 332 subzones in heat CSV -> 332 retained after pillar join
✅ No subzones dropped in the pillar join.
⚠️  318 subzones missing from held-out LST table — RMSE computed on the 14-subzone overlap only.


## RI.3 — Score construction

PCA loading sign is aligned on the exposure loading only — a pillar whose
loading disagrees with exposure can come out negative, which *inverts* its
contribution to the score rather than just down-weighting it. That's a
methodological call for whoever owns S6's weighting, not something this
notebook resolves silently — the printed weights per variant (RI.5) make it
visible when it happens.


In [7]:
# --- RI CELL 4: Score builder -------------------------------------------------
from sklearn.decomposition import PCA

def build_score(df: pd.DataFrame, exposure_col: str, weighting: str):
    """
    Rebuild the cooling-priority score for ONE heat-layer variant.
    Exposure = exposure_col (the variant under test).
    Sensitivity and Adaptive capacity are FROZEN (identical inputs across variants).
    weighting: "pca" (refit fresh here, per-variant) or "equal".
    """
    exposure_norm = normalize(df[exposure_col])
    sensitivity_norm = normalize(df["sensitivity_raw"])
    adaptive_deficit_norm = normalize(1 - df["greenery_fraction"])

    pillars = pd.DataFrame({
        "exposure": exposure_norm,
        "sensitivity": sensitivity_norm,
        "adaptive_deficit": adaptive_deficit_norm,
    })

    if weighting == "equal":
        score = pillars.mean(axis=1)
        weights = {"exposure": 1 / 3, "sensitivity": 1 / 3, "adaptive_deficit": 1 / 3}

    elif weighting == "pca":
        pca = PCA(n_components=1, random_state=SEED)
        pca.fit(pillars.values)
        raw_weights = pca.components_[0]
        if raw_weights[0] < 0:
            raw_weights = -raw_weights
        weights_arr = raw_weights / raw_weights.sum()
        score = pd.Series(pillars.values @ weights_arr, index=pillars.index)
        weights = dict(zip(pillars.columns, weights_arr))

    else:
        raise ValueError(f"Unknown weighting '{weighting}', expected 'pca' or 'equal'.")

    return score, weights

print("build_score() defined.")


build_score() defined.


## RI.4 — Ablation metrics

In [8]:
# --- RI CELL 5: RMSE / Spearman / top-N overlap helpers ---------------------
from scipy.stats import spearmanr
from sklearn.metrics import mean_squared_error

def rmse_vs_heldout(df, exposure_col, heldout):
    if heldout is None:
        return np.nan
    merged = df[["subzone_id", exposure_col]].merge(
        heldout[["subzone_id", "lst_heldout_c"]], on="subzone_id", how="inner"
    )
    if len(merged) == 0:
        return np.nan
    return float(np.sqrt(mean_squared_error(merged["lst_heldout_c"], merged[exposure_col])))


def top_n_overlap(score_a: pd.Series, score_b: pd.Series, n=TOP_N):
    top_a = set(score_a.sort_values(ascending=False).index[:n])
    top_b = set(score_b.sort_values(ascending=False).index[:n])
    return len(top_a & top_b) / n

print("Ablation metric helpers defined.")


Ablation metric helpers defined.


## RI.5 — Run: build the rank-impact table

In [9]:
# --- RI CELL 6: Run -----------------------------------------------------------
print(f"Subzones scored: {len(df)} | weighting scheme: {WEIGHTING} | seed: {SEED}\n")

scores = {}
weights_by_variant = {}
for variant in VARIANT_COLUMNS:
    score, weights = build_score(df, variant, WEIGHTING)
    scores[variant] = score
    weights_by_variant[variant] = weights

ref_score = scores[REFERENCE_VARIANT]

rows = []
for variant in VARIANT_COLUMNS:
    rmse = rmse_vs_heldout(df, variant, heldout)
    corr, _ = spearmanr(scores[variant], ref_score)
    overlap = top_n_overlap(scores[variant], ref_score, TOP_N)
    rows.append({
        "variant": variant,
        "lst_rmse_heldout": rmse,
        f"spearman_vs_{REFERENCE_VARIANT}": corr,
        f"top{TOP_N}_overlap_vs_{REFERENCE_VARIANT}": overlap,
    })

results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False, float_format=lambda x: "NaN" if pd.isna(x) else f"{x:.3f}"))

print("\nWeights used per variant (exposure, sensitivity, adaptive_deficit):")
for variant, w in weights_by_variant.items():
    print(f"  {variant}: {', '.join(f'{k}={v:.3f}' for k, v in w.items())}")
    if any(v < 0 for v in w.values()):
        print(f"    ⚠️  negative weight present — see RI.3 note on PCA sign-alignment.")

print(f"\n(Note: {REFERENCE_VARIANT} row is self-compared — spearman=1.000, "
      f"overlap=1.000 by construction. It's the frozen reference, not a result.)")

results_df


Subzones scored: 332 | weighting scheme: pca | seed: 42

      variant  lst_rmse_heldout  spearman_vs_lst_native30  top20_overlap_vs_lst_native30
 lst_native30            11.710                     1.000                          1.000
lst_bicubic10            11.711                     1.000                          1.000
lst_regress10            11.715                     1.000                          1.000

Weights used per variant (exposure, sensitivity, adaptive_deficit):
  lst_native30: exposure=0.357, sensitivity=0.119, adaptive_deficit=0.524
  lst_bicubic10: exposure=0.357, sensitivity=0.119, adaptive_deficit=0.524
  lst_regress10: exposure=0.357, sensitivity=0.119, adaptive_deficit=0.524

(Note: lst_native30 row is self-compared — spearman=1.000, overlap=1.000 by construction. It's the frozen reference, not a result.)


,variant,lst_rmse_heldout,spearman_vs_lst_native30,top20_overlap_vs_lst_native30
0,lst_native30,11.710248,1.000000,1.0
1,lst_bicubic10,11.711243,0.999995,1.0
2,lst_regress10,11.715151,0.999989,1.0


## RI.6 — Verdict

In [10]:
# --- RI CELL 7: Verdict -------------------------------------------------------
print("\n--- RI Verdict ---")

ri_checks = {
    "Subzones scored (non-zero)": len(df) > 0,
    "No majority drop in pillar join": n_dropped < len(heat) * 0.5 if len(heat) else False,
    f"Reference row ({REFERENCE_VARIANT}) self-check == 1.000": abs(results_df.loc[results_df['variant'] == REFERENCE_VARIANT, f'spearman_vs_{REFERENCE_VARIANT}'].iloc[0] - 1.0) < 1e-9,
    "RMSE computed for at least one variant (if held-out supplied)": (heldout is None) or results_df["lst_rmse_heldout"].notna().any(),
}

for check, passed in ri_checks.items():
    print(f"  [{'PASS' if passed else 'FAIL'}] {check}")

ri_pass = all(ri_checks.values())
if TOY_MODE:
    print("\n⚠️  TOY_MODE is on — this run is wiring-only, not a reportable result "
          "regardless of PASS/FAIL below.")
if ri_pass:
    print("\n✅ RI PASS: rank-impact table built successfully.")
else:
    print("\n⚠️  RI FAIL: fix flagged step(s) above before trusting this table.")

rank_impact_results["RI_rank_impact"] = {
    "status": "PASS" if ri_pass else "FAIL",
    "checks": ri_checks,
    "toy_mode": TOY_MODE,
    "weighting": WEIGHTING,
    "n_subzones_scored": len(df),
    "n_subzones_dropped": n_dropped,
}



--- RI Verdict ---
  [PASS] Subzones scored (non-zero)
  [PASS] No majority drop in pillar join
  [PASS] Reference row (lst_native30) self-check == 1.000
  [PASS] RMSE computed for at least one variant (if held-out supplied)

✅ RI PASS: rank-impact table built successfully.


## RI.7 (optional) — Rerun under the other weighting scheme

Per the locked eval rules: PCA is the primary scoring method, equal-weight
is the mandatory sensitivity check. Re-running here keeps both results in
the same session for direct comparison instead of re-running the whole
notebook with WEIGHTING flipped in RI.1.


In [11]:
# --- RI CELL 8 (optional): Sensitivity-check run (equal-weight) -------------
OTHER_WEIGHTING = "equal" if WEIGHTING == "pca" else "pca"

scores_other = {}
for variant in VARIANT_COLUMNS:
    score, _ = build_score(df, variant, OTHER_WEIGHTING)
    scores_other[variant] = score

ref_score_other = scores_other[REFERENCE_VARIANT]

rows_other = []
for variant in VARIANT_COLUMNS:
    rmse = rmse_vs_heldout(df, variant, heldout)
    corr, _ = spearmanr(scores_other[variant], ref_score_other)
    overlap = top_n_overlap(scores_other[variant], ref_score_other, TOP_N)
    rows_other.append({
        "variant": variant,
        "lst_rmse_heldout": rmse,
        f"spearman_vs_{REFERENCE_VARIANT}": corr,
        f"top{TOP_N}_overlap_vs_{REFERENCE_VARIANT}": overlap,
    })

results_df_other = pd.DataFrame(rows_other)
print(f"Sensitivity-check run — weighting = {OTHER_WEIGHTING}:\n")
print(results_df_other.to_string(index=False, float_format=lambda x: "NaN" if pd.isna(x) else f"{x:.3f}"))
results_df_other


Sensitivity-check run — weighting = equal:

      variant  lst_rmse_heldout  spearman_vs_lst_native30  top20_overlap_vs_lst_native30
 lst_native30            11.710                     1.000                          1.000
lst_bicubic10            11.711                     1.000                          1.000
lst_regress10            11.715                     1.000                          1.000


,variant,lst_rmse_heldout,spearman_vs_lst_native30,top20_overlap_vs_lst_native30
0,lst_native30,11.710248,1.000000,1.0
1,lst_bicubic10,11.711243,0.999997,1.0
2,lst_regress10,11.715151,0.999976,1.0


## RI.8 — Save results to Drive

In [12]:
# --- RI CELL 9: Save results to Drive -----------------------------------------
OUT_PRIMARY = f"/content/drive/MyDrive/{EXPORT_FOLDER}/rank_impact_results_{WEIGHTING}.csv"
results_df.to_csv(OUT_PRIMARY, index=False)
print(f"Saved: {OUT_PRIMARY}")

try:
    OUT_OTHER = f"/content/drive/MyDrive/{EXPORT_FOLDER}/rank_impact_results_{OTHER_WEIGHTING}.csv"
    results_df_other.to_csv(OUT_OTHER, index=False)
    print(f"Saved: {OUT_OTHER}")
except NameError:
    print("(RI.7 sensitivity-check cell wasn't run — only the primary-weighting result was saved.)")


Saved: /content/drive/MyDrive/urban_heat_sg/rank_impact_results_pca.csv
Saved: /content/drive/MyDrive/urban_heat_sg/rank_impact_results_equal.csv
